# Week 4 — NumPy Fundamentals I
## Companion Notebook: Student Exam Score Analysis Pipeline

**Course:** Introduction to Data Science  
**Textbook:** McKinney, *Python for Data Analysis*, 3rd Ed. — Chapter 4 §4.1  

---

This notebook walks through a **complete, functioning pipeline** that mirrors real data science work.  
We build it step-by-step, introducing each NumPy concept from the lecture as we need it.

**Pipeline overview:**
```
Raw scores (list)  →  ndarray  →  inspect attributes  →  arithmetic  →  indexing  →  summary report
```

---
## Setup

In [1]:
import numpy as np

# Always check your NumPy version
print(f'NumPy version: {np.__version__}')

NumPy version: 2.2.6


---
## Part 1 — Why NumPy? The Performance Argument

Before using NumPy, let's see *why* it matters.

In [2]:
import time

N = 1_000_000

# ── Approach 1: Python list ───────────────────────────────────────────────
py_list = list(range(N))

t0 = time.perf_counter()
result_list = [x * 2 for x in py_list]
t1 = time.perf_counter()
list_time = t1 - t0

# ── Approach 2: NumPy array ───────────────────────────────────────────────
np_array = np.arange(N)

t0 = time.perf_counter()
result_array = np_array * 2
t1 = time.perf_counter()
numpy_time = t1 - t0

print(f'Python list:  {list_time*1000:.1f} ms')
print(f'NumPy array:  {numpy_time*1000:.1f} ms')
print(f'Speedup:      {list_time/numpy_time:.0f}×')

Python list:  61.7 ms
NumPy array:  4.6 ms
Speedup:      13×


> **What you should see:** NumPy is typically 50–100× faster. The speedup comes from vectorised C operations applied to a contiguous block of memory — no Python loop overhead.

---
## Part 2 — Creating the Dataset

We have exam scores for **60 students** across **3 subjects**: Mathematics, Statistics, and Programming.  
Scores are integers from 0–100.

In [3]:
# Seed for reproducibility — same random numbers every run
rng = np.random.default_rng(seed=42)

# ── Generate synthetic exam scores ───────────────────────────────────────
# Shape: (60 students, 3 subjects)
# Subject means: Math=72, Stats=68, Programming=75
n_students = 60
n_subjects = 3
subject_names = ['Mathematics', 'Statistics', 'Programming']

means = np.array([72, 68, 75])      # one mean per subject
stds  = np.array([12, 14, 11])      # one std per subject

# Generate scores: shape (60, 3)
raw_scores = rng.normal(loc=means, scale=stds, size=(n_students, n_subjects))

# Clip to valid range [0, 100] and round to nearest integer
scores = np.clip(np.round(raw_scores), 0, 100).astype(np.int32)

print('scores shape:', scores.shape)   # (60, 3)
print('scores dtype:', scores.dtype)   # int32
print('First 5 rows:')
print(scores[:5])

scores shape: (60, 3)
scores dtype: int32
First 5 rows:
[[76 53 83]
 [83 41 61]
 [74 64 75]
 [62 80 84]
 [73 84 80]]


---
## Part 3 — Inspecting Array Attributes

The four key attributes: **shape, ndim, size, dtype**

In [4]:
print('=== Array Attributes ===')
print(f'  shape : {scores.shape}   ← (rows=students, cols=subjects)')
print(f'  ndim  : {scores.ndim}              ← 2D array')
print(f'  size  : {scores.size}           ← total elements (60 × 3)')
print(f'  dtype : {scores.dtype}           ← 32-bit integer')
print(f'  nbytes: {scores.nbytes} bytes     ← {scores.nbytes / 1024:.2f} KB in memory')

=== Array Attributes ===
  shape : (60, 3)   ← (rows=students, cols=subjects)
  ndim  : 2              ← 2D array
  size  : 180           ← total elements (60 × 3)
  dtype : int32           ← 32-bit integer
  nbytes: 720 bytes     ← 0.70 KB in memory


In [5]:
# ── Memory comparison: int32 vs float64 ──────────────────────────────────
scores_f64 = scores.astype(np.float64)

print('Memory comparison:')
print(f'  int32   : {scores.nbytes} bytes')
print(f'  float64 : {scores_f64.nbytes} bytes  ← 2× larger')
print()
print('astype() creates a NEW array — original is unchanged:')
print(f'  scores.dtype still: {scores.dtype}')

Memory comparison:
  int32   : 720 bytes
  float64 : 1440 bytes  ← 2× larger

astype() creates a NEW array — original is unchanged:
  scores.dtype still: int32


---
## Part 4 — Array Creation Patterns

Demonstrating the main creation functions on meaningful examples.

In [6]:
# ── Grade boundaries ─────────────────────────────────────────────────────
grade_boundaries = np.array([0, 60, 70, 80, 90, 100])
print('Grade boundaries:', grade_boundaries)

# ── Student IDs ──────────────────────────────────────────────────────────
student_ids = np.arange(1, n_students + 1)    # [1, 2, 3, ..., 60]
print('Student IDs (first 10):', student_ids[:10])

# ── Weight vector for weighted average ───────────────────────────────────
# Math 40%, Stats 30%, Programming 30%
weights = np.array([0.40, 0.30, 0.30])
print('Weights:', weights)
print('Weights sum to:', weights.sum())   # must be 1.0

# ── Initialise output arrays with zeros ──────────────────────────────────
weighted_avg = np.zeros(n_students)       # one value per student
pass_flags   = np.zeros(n_students, dtype=np.bool_)  # bool array
print('\nInitial weighted_avg:', weighted_avg[:5])
print('Initial pass_flags  :', pass_flags[:5])

Grade boundaries: [  0  60  70  80  90 100]
Student IDs (first 10): [ 1  2  3  4  5  6  7  8  9 10]
Weights: [0.4 0.3 0.3]
Weights sum to: 1.0

Initial weighted_avg: [0. 0. 0. 0. 0.]
Initial pass_flags  : [False False False False False]


In [7]:
# ── linspace for a grade scale plot ──────────────────────────────────────
score_axis = np.linspace(0, 100, 101)   # 0, 1, 2, ..., 100
print('Score axis (first 5):', score_axis[:5])
print('Score axis (last 5) :', score_axis[-5:])

# ── eye for a correlation identity baseline ───────────────────────────────
identity = np.eye(n_subjects)
print('\nIdentity matrix (3×3):')
print(identity)

Score axis (first 5): [0. 1. 2. 3. 4.]
Score axis (last 5) : [ 96.  97.  98.  99. 100.]

Identity matrix (3×3):
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


---
## Part 5 — Vectorised Arithmetic

Computing statistics **without a single loop**.

In [8]:
# ── Per-subject statistics (axis=0 = collapse rows, keep columns) ─────────
subject_mean = scores.mean(axis=0)   # shape (3,) — one mean per subject
subject_std  = scores.std(axis=0)
subject_min  = scores.min(axis=0)
subject_max  = scores.max(axis=0)

print('Per-subject statistics:')
print(f'  {"Subject":<14} {"Mean":>6} {"Std":>6} {"Min":>5} {"Max":>5}')
print(f'  {"-"*40}')
for i, name in enumerate(subject_names):
    print(f'  {name:<14} {subject_mean[i]:>6.1f} {subject_std[i]:>6.1f} '
          f'{subject_min[i]:>5} {subject_max[i]:>5}')

Per-subject statistics:
  Subject          Mean    Std   Min   Max
  ----------------------------------------
  Mathematics      71.8   10.6    52    98
  Statistics       66.1   12.4    38   100
  Programming      74.7    8.6    57    94


In [9]:
# ── Weighted average per student ──────────────────────────────────────────
# scores shape:  (60, 3)
# weights shape: (3,)  ← broadcasts across rows automatically
weighted_avg = (scores * weights).sum(axis=1)   # shape (60,)

print('Weighted averages (first 10 students):')
print(np.round(weighted_avg[:10], 1))

print(f'\nOverall class mean: {weighted_avg.mean():.1f}')
print(f'Highest average   : {weighted_avg.max():.1f}')
print(f'Lowest average    : {weighted_avg.min():.1f}')

Weighted averages (first 10 students):
[71.2 63.8 71.3 74.  78.4 65.9 75.2 73.  70.  76.6]

Overall class mean: 71.0
Highest average   : 92.4
Lowest average    : 58.8


In [10]:
# ── Z-score normalisation — no loops! ────────────────────────────────────
# z = (x - mean) / std  applied element-wise across entire array
z_scores = (scores - subject_mean) / subject_std   # broadcasting

print('Z-scores shape:', z_scores.shape)  # still (60, 3)
print('Z-score means (should be ~0):', z_scores.mean(axis=0).round(6))
print('Z-score stds  (should be ~1):', z_scores.std(axis=0).round(6))

Z-scores shape: (60, 3)
Z-score means (should be ~0): [ 0. -0. -0.]
Z-score stds  (should be ~1): [1. 1. 1.]


In [11]:
# ── Normalise to 0-1 range (min-max scaling) ──────────────────────────────
s_min = scores.min(axis=0)    # shape (3,)
s_max = scores.max(axis=0)    # shape (3,)

normalised = (scores - s_min) / (s_max - s_min)   # all broadcasting
print('Normalised range per subject:')
print('  min:', normalised.min(axis=0).round(4))
print('  max:', normalised.max(axis=0).round(4))

Normalised range per subject:
  min: [0. 0. 0.]
  max: [1. 1. 1.]


---
## Part 6 — Indexing and Slicing

Selecting meaningful subsets of the data.

In [12]:
# ── Basic 2D indexing ─────────────────────────────────────────────────────
print('Student 0, Math score :', scores[0, 0])
print('Student 0, all scores :', scores[0])
print('All students, Math col:', scores[:, 0][:10], '...')

# ── First 5 students, all subjects ───────────────────────────────────────
print('\nFirst 5 students:')
print(scores[:5])

Student 0, Math score : 76
Student 0, all scores : [76 53 83]
All students, Math col: [76 83 74 62 73 62 83 64 67 76] ...

First 5 students:
[[76 53 83]
 [83 41 61]
 [74 64 75]
 [62 80 84]
 [73 84 80]]


In [13]:
# ── Views vs copies — the critical concept ────────────────────────────────
print('=== Views vs Copies ===')

arr = np.arange(10)
print('Original:', arr)

# Slice = VIEW
view = arr[2:6]
view[0] = 999
print('After modifying view[0]=999, original arr:', arr)
# arr[2] is now 999 — slice shares memory!

# Reset
arr = np.arange(10)

# .copy() = independent array
copy = arr[2:6].copy()
copy[0] = 999
print('After modifying copy[0]=999, original arr:', arr)
# arr is unchanged — copy is independent

=== Views vs Copies ===
Original: [0 1 2 3 4 5 6 7 8 9]
After modifying view[0]=999, original arr: [  0   1 999   3   4   5   6   7   8   9]
After modifying copy[0]=999, original arr: [0 1 2 3 4 5 6 7 8 9]


In [14]:
# ── Select top performers ─────────────────────────────────────────────────
# Students whose weighted average >= 85 (boolean indexing preview)
pass_threshold = 60
top_threshold  = 85

passed  = weighted_avg >= pass_threshold   # boolean array
top     = weighted_avg >= top_threshold

print(f'Pass threshold ({pass_threshold}): {passed.sum()} / {n_students} students')
print(f'Top threshold  ({top_threshold}): {top.sum()} / {n_students} students')

# Get the actual scores of top students
top_scores = scores[top]   # boolean indexing selects rows where True
print(f'\nTop students scores (shape: {top_scores.shape}):')
print(top_scores)

Pass threshold (60): 57 / 60 students
Top threshold  (85): 1 / 60 students

Top students scores (shape: (1, 3)):
[[ 96 100  80]]


---
## Part 7 — Grade Assignment

Using vectorised operations and `np.digitize` to assign letter grades.

In [15]:
# ── Assign letter grades based on weighted average ────────────────────────
# A: 90+   B: 80-89   C: 70-79   D: 60-69   F: <60

boundaries = np.array([60, 70, 80, 90])   # bin edges
grade_labels = np.array(['F', 'D', 'C', 'B', 'A'])

# np.digitize returns which bin each value falls into
grade_indices = np.digitize(weighted_avg, boundaries)
grades = grade_labels[grade_indices]

print('Grade distribution:')
for grade in ['A', 'B', 'C', 'D', 'F']:
    count = (grades == grade).sum()
    bar   = '█' * count
    print(f'  {grade}: {bar} ({count})')

Grade distribution:
  A: █ (1)
  B: ███ (3)
  C: █████████████████████████████████ (33)
  D: ████████████████████ (20)
  F: ███ (3)


---
## Part 8 — Final Summary Report

Generating a complete analytical summary using only NumPy.

In [16]:
# ── Correlation between subjects ──────────────────────────────────────────
# np.corrcoef expects (n_variables, n_observations) — need to transpose
corr_matrix = np.corrcoef(scores.T)   # scores.T shape: (3, 60)

print('=== Subject Correlation Matrix ===')
print(f'  {"":14}', end='')
for name in subject_names: print(f'{name:>14}', end='')
print()
for i, name in enumerate(subject_names):
    print(f'  {name:<14}', end='')
    for j in range(n_subjects):
        print(f'{corr_matrix[i,j]:>14.3f}', end='')
    print()

=== Subject Correlation Matrix ===
                   Mathematics    Statistics   Programming
  Mathematics            1.000        -0.039        -0.048
  Statistics            -0.039         1.000         0.288
  Programming           -0.048         0.288         1.000


In [17]:
# ── Percentile statistics ─────────────────────────────────────────────────
percentiles = [25, 50, 75, 90]
pct_values  = np.percentile(weighted_avg, percentiles)

print('=== Weighted Average Percentiles ===')
for p, v in zip(percentiles, pct_values):
    print(f'  {p:3d}th percentile: {v:.1f}')

=== Weighted Average Percentiles ===
   25th percentile: 65.9
   50th percentile: 71.0
   75th percentile: 75.2
   90th percentile: 78.5


In [18]:
# ── Full summary report ───────────────────────────────────────────────────
print('=' * 55)
print('  EXAM RESULTS SUMMARY — Introduction to Data Science')
print('=' * 55)
print(f'  Students analysed : {n_students}')
print(f'  Subjects          : {", ".join(subject_names)}')
print(f'  Weights           : 40% / 30% / 30%')
print()
print('  Per-Subject Performance:')
for i, name in enumerate(subject_names):
    print(f'    {name:<14}: mean={subject_mean[i]:.1f}  '
          f'std={subject_std[i]:.1f}  '
          f'range=[{subject_min[i]}, {subject_max[i]}]')
print()
print('  Overall Weighted Average:')
print(f'    Mean   : {weighted_avg.mean():.1f}')
print(f'    Median : {np.median(weighted_avg):.1f}')
print(f'    Std    : {weighted_avg.std():.1f}')
print(f'    Range  : [{weighted_avg.min():.1f}, {weighted_avg.max():.1f}]')
print()
print('  Grade Distribution:')
for grade in ['A', 'B', 'C', 'D', 'F']:
    count = (grades == grade).sum()
    pct   = count / n_students * 100
    print(f'    {grade}: {count:3d} students ({pct:4.1f}%)')
print()
print(f'  Pass rate (≥60): {passed.sum()}/{n_students} '
      f'({passed.mean()*100:.1f}%)')
print(f'  Top performers (≥85): {top.sum()}/{n_students} '
      f'({top.mean()*100:.1f}%)')
print('=' * 55)

  EXAM RESULTS SUMMARY — Introduction to Data Science
  Students analysed : 60
  Subjects          : Mathematics, Statistics, Programming
  Weights           : 40% / 30% / 30%

  Per-Subject Performance:
    Mathematics   : mean=71.8  std=10.6  range=[52, 98]
    Statistics    : mean=66.1  std=12.4  range=[38, 100]
    Programming   : mean=74.7  std=8.6  range=[57, 94]

  Overall Weighted Average:
    Mean   : 71.0
    Median : 71.0
    Std    : 6.4
    Range  : [58.8, 92.4]

  Grade Distribution:
    A:   1 students ( 1.7%)
    B:   3 students ( 5.0%)
    C:  33 students (55.0%)
    D:  20 students (33.3%)
    F:   3 students ( 5.0%)

  Pass rate (≥60): 57/60 (95.0%)
  Top performers (≥85): 1/60 (1.7%)


---
## Part 9 — Exercises (Solutions)

These solutions are distributed after class. Work through the exercises yourself first before reviewing.


In [19]:
# ── Exercise 1: ndarray attributes ──────────────────────────────────────────
# Create a 1D array of 15 evenly-spaced values from 0 to 1.
# Print its shape, ndim, size, and dtype.

arr = np.linspace(0, 1, 15)

print('Array:      ', arr)
print('shape:      ', arr.shape)   # (15,)
print('ndim:       ', arr.ndim)    # 1
print('size:       ', arr.size)    # 15
print('dtype:      ', arr.dtype)   # float64  -- linspace defaults to float64


Array:       [0.         0.07142857 0.14285714 0.21428571 0.28571429 0.35714286
 0.42857143 0.5        0.57142857 0.64285714 0.71428571 0.78571429
 0.85714286 0.92857143 1.        ]
shape:       (15,)
ndim:        1
size:        15
dtype:       float64


In [20]:
# ── Exercise 2: Array creation ───────────────────────────────────────────────
# (a) Create a 4x4 identity matrix using np.eye().
# (b) Create a 3x5 matrix of all sevens using np.full().
# Print both arrays and inspect their shapes.

# (a) 4x4 identity matrix
identity = np.eye(4)
print('4x4 identity matrix:')
print(identity)
print('shape:', identity.shape)   # (4, 4)
print()

# (b) 3x5 matrix filled with 7.0
sevens = np.full((3, 5), 7.0)
print('3x5 matrix of sevens:')
print(sevens)
print('shape:', sevens.shape)     # (3, 5)


4x4 identity matrix:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
shape: (4, 4)

3x5 matrix of sevens:
[[7. 7. 7. 7. 7.]
 [7. 7. 7. 7. 7.]
 [7. 7. 7. 7. 7.]]
shape: (3, 5)


In [21]:
# ── Exercise 3: Data types (dtype) ───────────────────────────────────────────
# (a) Create np.array([1, 2, 3]) and check its dtype.
# (b) Create the same array with dtype=np.float32.
# (c) Compare memory usage of both using .nbytes.

# (a) Default dtype
arr_default = np.array([1, 2, 3])
print('Default dtype: ', arr_default.dtype)    # int64  (on most 64-bit systems)
print('nbytes:        ', arr_default.nbytes)   # 24  (3 elements x 8 bytes each)
print()

# (b) Explicit float32
arr_f32 = np.array([1, 2, 3], dtype=np.float32)
print('float32 dtype: ', arr_f32.dtype)        # float32
print('nbytes:        ', arr_f32.nbytes)        # 12  (3 elements x 4 bytes each)
print()

# (c) Comparison
print(f'int64   : {arr_default.nbytes} bytes  -- {arr_default.nbytes // len(arr_default)} bytes per element')
print(f'float32 : {arr_f32.nbytes} bytes  -- {arr_f32.nbytes // len(arr_f32)} bytes per element')
print(f'Memory saving: {arr_default.nbytes - arr_f32.nbytes} bytes ({(1 - arr_f32.nbytes/arr_default.nbytes)*100:.0f}% smaller)')


Default dtype:  int64
nbytes:         24

float32 dtype:  float32
nbytes:         12

int64   : 24 bytes  -- 8 bytes per element
float32 : 12 bytes  -- 4 bytes per element
Memory saving: 12 bytes (50% smaller)


In [22]:
# ── Exercise 4: Vectorised arithmetic ───────────────────────────────────────
scores = np.array([72, 85, 91, 68, 79, 95])

# (a) Mean
mean_score = scores.mean()
print(f'(a) Mean score: {mean_score:.2f}')           # 81.67
print()

# (b) Scores above 80 -- boolean indexing
above_80 = scores[scores > 80]
print(f'(b) Scores above 80: {above_80}')             # [85 91 95]
print(f'    Boolean mask:     {scores > 80}')         # [F T T F F T]
print()

# (c) Min-max normalisation to 0-100
scaled = (scores - scores.min()) / (scores.max() - scores.min()) * 100
print(f'(c) Scaled scores (0-100 range):')
print(f'    Original : {scores}')
print(f'    Scaled   : {np.round(scaled, 1)}')
print(f'    Min: {scaled.min():.1f}  Max: {scaled.max():.1f}')  # 0.0  100.0


(a) Mean score: 81.67

(b) Scores above 80: [85 91 95]
    Boolean mask:     [False  True  True False False  True]

(c) Scaled scores (0-100 range):
    Original : [72 85 91 68 79 95]
    Scaled   : [ 14.8  63.   85.2   0.   40.7 100. ]
    Min: 0.0  Max: 100.0


In [23]:
# ── Exercise 5: 2D indexing ──────────────────────────────────────────────────
arr = np.arange(12).reshape(3, 4)
print('Array:')
print(arr)
print()

# (a) Second row  -- row index 1
second_row = arr[1]
print('(a) Second row (arr[1]):')
print('   ', second_row)                             # [4 5 6 7]
print()

# (b) Last column  -- all rows, column index -1
last_col = arr[:, -1]
print('(b) Last column (arr[:, -1]):')
print('   ', last_col)                               # [ 3  7 11]
print()

# (c) 2x2 submatrix in the bottom-right corner
#     Rows 1-2 (last two rows), Cols 2-3 (last two columns)
submatrix = arr[1:, 2:]
print('(c) Bottom-right 2x2 submatrix (arr[1:, 2:]):')
print(submatrix)                                     # [[ 6  7]
print()                                              #  [10 11]]
print('Verification -- shape:', submatrix.shape)     # (2, 2)


Array:
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

(a) Second row (arr[1]):
    [4 5 6 7]

(b) Last column (arr[:, -1]):
    [ 3  7 11]

(c) Bottom-right 2x2 submatrix (arr[1:, 2:]):
[[ 6  7]
 [10 11]]

Verification -- shape: (2, 2)


---
## NumPy Quick Reference

| Concept | Code | Notes |
|---|---|---|
| Create from list | `np.array([1,2,3])` | dtype inferred |
| Zeros / ones | `np.zeros((3,4))` | float64 default |
| Range | `np.arange(0,10,2)` | like `range()` |
| Evenly spaced | `np.linspace(0,1,5)` | includes endpoint |
| Identity | `np.eye(3)` | 3×3 identity matrix |
| Inspect shape | `arr.shape` | tuple |
| Total elements | `arr.size` | rows × cols × ... |
| Change dtype | `arr.astype(np.float32)` | returns new array |
| Column stats | `arr.mean(axis=0)` | axis=0 → per column |
| Row stats | `arr.mean(axis=1)` | axis=1 → per row |
| Slice (view) | `arr[1:4]` | shares memory! |
| Copy | `arr[1:4].copy()` | independent |
| 2D element | `arr[row, col]` | faster than `arr[row][col]` |
| Column | `arr[:, col]` | all rows, one col |
| Submatrix | `arr[r1:r2, c1:c2]` | row and col slices |